# 📝 Notebook 2: Text Recognition (CRNN)

Train CRNN to recognize text in cropped regions.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.model import CRNN
from src.dataset import crop_text_regions, TextDataset, get_transforms
from src.train import fit
from src.utils import CHAR_TO_IDX, VOCAB_SIZE, plot_losses, show_predictions

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Crop text regions
DATASET_DIR = "../datasets/SceneTrialTrain"

paths, labels = crop_text_regions(DATASET_DIR, "../cropped_text")
print(f"Cropped {len(paths)} text regions")

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(paths, labels, test_size=0.2, random_state=42)

max_len = max(len(l) for l in labels)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Max length: {max_len}")

In [ ]:
# Create datasets
transforms = get_transforms()

train_ds = TextDataset(X_train, y_train, CHAR_TO_IDX, max_len, transforms["train"])
val_ds = TextDataset(X_val, y_val, CHAR_TO_IDX, max_len, transforms["val"])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

In [ ]:
# Create model
model = CRNN(vocab_size=VOCAB_SIZE).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {params:,}")

In [ ]:
# Train
train_losses, val_losses = fit(
    model, train_loader, val_loader,
    epochs=80, lr=1e-3, device=device
)

In [ ]:
# Plot losses
plot_losses(train_losses, val_losses)

In [ ]:
# Show predictions
show_predictions(model, val_ds, device)

In [ ]:
# Save model
torch.save(model.state_dict(), "../crnn.pt")
print("Model saved to crnn.pt")

✅ **Done!** CRNN model saved to `crnn.pt`

➡️ Next: Run `03_evaluate.ipynb`